# Multi-Agent Systems Engineering Platform

**Agent 1 — Operational Agent**: Drives a structured SE interview with the user, then generates a requirements JSON (ISO-aligned).

**Agent 2 — Research Agent**: Reads Agent 1's JSON, validates understanding with user, searches ArXiv + DuckDuckGo, filters & ranks solutions.

**Requirements**
```
pip install langchain langchain-anthropic langchain-community pydantic httpx python-dotenv ipywidgets
```
Set `ANTHROPIC_API_KEY` in your `.env` file.

---
## CELL 1 — Installs & Imports

In [21]:
import os, json, re, time, textwrap, urllib.parse, urllib.request, ssl, xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Optional, Dict, Any
from enum import Enum
from uuid import uuid4

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# LangChain
from langchain_anthropic import ChatAnthropic
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.memory import ConversationBufferMemory
from langchain_core.chat_history import InMemoryChatMessageHistory

# Load env
for d in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (d / '.env').exists():
        load_dotenv(d / '.env')
        print(f'✓ Loaded .env from {d}')
        break

ANTHROPIC_KEY = os.getenv('ANTHROPIC_API_KEY', '')
assert ANTHROPIC_KEY, 'Set ANTHROPIC_API_KEY in your .env'

MODEL      = 'claude-sonnet-4-20250514'
MAX_TOKENS = 4096
TEMP       = 0.3

print(f'✓ Model: {MODEL}')
print('✓ All imports OK')

✓ Loaded .env from c:\Projects\poc_v1
✓ Model: claude-sonnet-4-20250514
✓ All imports OK


---
## CELL 2 — Shared Data Models

In [22]:
class GapSeverity(str, Enum):
    COVERED  = 'covered'
    PARTIAL  = 'partial'
    GAP      = 'gap'
    NO_MATCH = 'no_match'

class TRL(str, Enum):
    TRL1='TRL 1'; TRL2='TRL 2'; TRL3='TRL 3'; TRL4='TRL 4'; TRL5='TRL 5'
    TRL6='TRL 6'; TRL7='TRL 7'; TRL8='TRL 8'; TRL9='TRL 9'; UNKNOWN='unknown'

class StandardMatch(BaseModel):
    name: str
    clause: Optional[str] = None
    verbatim_excerpt: Optional[str] = None
    similarity_score: float = 0.0
    issuing_body: Optional[str] = None
    authority_level: Optional[str] = None
    source_url: Optional[str] = None
    year: Optional[str] = None

class TechnologyMatch(BaseModel):
    name: str
    description: str = ''
    source: Optional[str] = None
    domain: Optional[str] = None
    trl: TRL = TRL.UNKNOWN
    feasibility: str = 'Unknown'
    realism: str = 'Unknown'
    score: float = 0.0
    notes: Optional[str] = None

class ResearchRecord(BaseModel):
    req_id: str
    req_statement: str
    domain_tag: Optional[str] = None
    criticality: str = 'medium'
    standards: List[StandardMatch] = Field(default_factory=list)
    technologies: List[TechnologyMatch] = Field(default_factory=list)
    gap_severity: GapSeverity = GapSeverity.NO_MATCH
    gap_description: str = ''
    recommendation: str = ''
    researched_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

# Global state — Agent 1 writes, Agent 2 reads
AGENT1_OUTPUT: Optional[Dict] = None   # the generated requirements JSON
AGENT1_CHAT_HISTORY: List[Dict[str, str]] = []  # [{"role":"user|assistant","content":"..."}, ...]
AGENT2_RECORDS: List[ResearchRecord] = []

print('✓ Data models ready')

✓ Data models ready


---
## CELL 3 — LLM Factory

In [23]:
def make_llm(temperature=TEMP) -> ChatAnthropic:
    return ChatAnthropic(
        model=MODEL,
        api_key=ANTHROPIC_KEY,
        max_tokens=MAX_TOKENS,
        temperature=temperature,
    )

print('✓ make_llm() ready')

✓ make_llm() ready


---
## CELL 4 — Agent 1: Operational Agent (SE Interview + JSON Generation)

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 1 — OPERATIONAL AGENT
# Drives a structured Systems Engineering interview across 6 SE phases,
# then calls the LLM to generate a rich requirements JSON.
# Uses LangChain ConversationBufferMemory for full context retention.
# ─────────────────────────────────────────────────────────────────────────────

# SE interview phases — each phase has a trigger keyword and a guiding prompt
SE_PHASES = [
    {
        'id': 'intake',
        'label': 'Intake & Stakeholder Needs',
        'question': (
            'Great! Let\'s start with **Stakeholder Needs**.\n'
            'Please describe your project and what your stakeholders expect from this system. '
            'Who are the key stakeholders, and what are their primary needs or pain points?'
        )
    },
    {
        'id': 'context',
        'label': 'System Context',
        'question': (
            'Understood. Now let\'s define the **System Context**.\n'
            'What is the broader world your system lives in? What domain (e.g. nuclear, aerospace, medical)? '
            'What problem does it ultimately solve?'
        )
    },
    {
        'id': 'environment',
        'label': 'Operating Environment',
        'question': (
            'Good. Now describe the **Operating Environment**.\n'
            'Where and how will the system operate? Consider physical conditions '
            '(temperature, radiation, pressure), organizational context (regulations, standards), '
            'and technical infrastructure (networks, power, adjacent systems).'
        )
    },
    {
        'id': 'boundaries',
        'label': 'System Boundaries',
        'question': (
            'Let\'s define the **System Boundaries**.\n'
            'What is IN scope — i.e., what must this system do or include?\n'
            'What is explicitly OUT of scope — what should this system NOT do or handle?'
        )
    },
    {
        'id': 'actors',
        'label': 'Actors & External Systems',
        'question': (
            'Now identify **Actors & External Systems**.\n'
            'Who or what interacts with your system? List human operators, regulators, '
            'external software systems, hardware, or other subsystems your system must interface with.'
        )
    },
    {
        'id': 'interfaces',
        'label': 'Interfaces at the Boundary',
        'question': (
            'Finally, describe the **Interfaces at the Boundary**.\n'
            'How do those external actors and systems interact with yours? '
            'What data, signals, or physical exchanges happen at the boundary? '
            'Any specific protocols, standards, or interface requirements?'
        )
    },
]


# Dynamic system prompt for the JSON generation step

def build_generation_prompt(conversation_summary: str, iso_context: str = '') -> str:
    return f"""You are an expert Systems Engineer who has just completed a structured
requirements elicitation interview with a user.

REFERENCE CONTEXT (optional):
{iso_context if iso_context else 'Use Systems Engineering elicitation best-practices. Build a structured model from stakeholder needs, context, environment, boundaries, actors, interfaces, and lifecycle/scenario thinking.'}

CONVERSATION CAPTURED:
{conversation_summary}

YOUR TASK:
Generate a comprehensive, structured requirements JSON from this conversation.
The JSON must follow this exact schema:

{{
  "meta": {{
    "standard": "<inferred standard(s) if mentioned, else 'not_specified'>",
    "project_name": "<inferred from conversation>",
    "domain": "<primary domain>",
    "generated_at": "<ISO timestamp>",
    "context_summary": "<2-3 sentence project summary>"
  }},
  "iso15926_model": {{
    "meta": {{
      "standard": "<inferred if applicable, else 'not_specified'>",
      "domain": "<domain>",
      "active_domains": ["<domain1>", "<domain2>"]
    }},
    "possible_individuals": [
      {{
        "class": "stakeholder_need | regulatory_clause | engineering_constraint",
        "name": "<short requirement name>",
        "description": "<the system SHALL... statement>",
        "priority": "high | medium | low",
        "is_assumption": false,
        "domain_tag": "<domain>",
        "rationale": "<why this requirement exists>",
        "source": "stakeholder_interview"
      }}
    ],
    "classes": [
      {{
        "class": "operational | failure | maintenance | degraded | lifecycle",
        "name": "<scenario name>",
        "description": "<detailed scenario description>",
        "priority": "high | medium | low",
        "is_assumption": false,
        "domain_tag": "<domain>"
      }}
    ],
    "things": [
      {{
        "class": "actor | interface | boundary_element",
        "name": "<element name>",
        "description": "<description>",
        "is_assumption": false,
        "domain_tag": "<domain>"
      }}
    ]
  }},
  "system_context": {{
    "environment": "<operating environment summary>",
    "in_scope": ["<item1>", "<item2>"],
    "out_of_scope": ["<item1>"],
    "actors": ["<actor1>", "<actor2>"],
    "interfaces": ["<interface1>"]
  }}
}}

RULES:
- Generate at least 8 requirements across the possible_individuals array
- Generate at least 6 operational/lifecycle scenarios in the classes array  
- Infer reasonable assumptions where user input is sparse (mark is_assumption: true)
- Use SHALL language for requirements
- Assign domain_tags that reflect the user's domain (not hardcoded nuclear unless they said so)
- Return ONLY valid JSON, no preamble or markdown fences
"""


class OperationalAgent:
    """
    Agent 1: Drives SE interview via a chat loop.
    Uses LangChain ConversationBufferMemory to maintain full conversation context.
    """

    def __init__(self, iso_reference_path: Optional[str] = None):
        self.llm = make_llm(temperature=0.4)
        self.memory = ConversationBufferMemory(
            return_messages=True,
            memory_key='chat_history',
            human_prefix='User',
            ai_prefix='OperationalAgent'
        )
        self.phase_idx = 0
        self.phase_answers: Dict[str, str] = {}
        self.done = False
        self.iso_context = self._load_iso(iso_reference_path)

    def _load_iso(self, path: Optional[str]) -> str:
        if not path:
            return ''
        try:
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
            # Extract key meta for dynamic context injection
            meta = data.get('iso15926_model', {}).get('meta', {})
            return f'Standard: {meta.get("standard","ISO 15926")} | Domain: {meta.get("domain","")}'
        except Exception as e:
            return f'ISO 15926 ontology reference (could not load: {e})'

    def greeting(self) -> str:
        return (
            '👋 Welcome to the **Systems Engineering Requirements Platform**.\n\n'
            'I am your **Operational Agent**. I will guide you through a structured '
            'SE elicitation interview covering:\n'
            '1. Stakeholder Needs\n2. System Context\n3. Operating Environment\n'
            '4. System Boundaries\n5. Actors & External Systems\n6. Interfaces\n\n'
            'Once we\'re done, I will generate a full ISO 15926-aligned requirements JSON, '
            'which the **Research Agent** will use for the next phase.\n\n'
            'Let\'s begin! Please start by giving me a brief project description.'
        )

    def _current_phase(self) -> Optional[Dict]:
        if self.phase_idx < len(SE_PHASES):
            return SE_PHASES[self.phase_idx]
        return None

    def chat(self, user_input: str) -> str:
        """Process one user turn. Returns agent reply."""
        if self.done:
            return '✅ Requirements elicitation complete. Please run Agent 2 for the research phase.'

        phase = self._current_phase()
        if not phase:
            return self._finalize()

        # Store the user's answer for this phase
        self.phase_answers[phase['id']] = user_input

        # Save to LangChain memory
        self.memory.save_context(
            {'input': f'[{phase["label"]}] {user_input}'},
            {'output': ''}
        )

        # Move to next phase
        self.phase_idx += 1
        next_phase = self._current_phase()

        if next_phase:
            # Ask a contextually-aware follow-up using the LLM
            reply = self._contextual_transition(user_input, phase, next_phase)
        else:
            reply = self._finalize()

        return reply

    def _contextual_transition(self, user_answer: str, current_phase: Dict, next_phase: Dict) -> str:
        """Use LLM to acknowledge the answer and ask the next phase question naturally."""
        history = self.memory.load_memory_variables({}).get('chat_history', [])
        history_str = '\n'.join([f'{m.type}: {m.content}' for m in history[-6:]])  # last 3 turns

        prompt = f"""You are a senior Systems Engineer conducting a requirements interview.
Recent conversation:
{history_str}

The user just answered the '{current_phase['label']}' phase: "{user_answer[:300]}"

Write a SHORT acknowledgment (1-2 sentences) that shows you understood their answer,
then naturally transition to asking about '{next_phase['label']}'.

Then ask this question (adapt it to fit the user's domain context naturally):
{next_phase['question']}

Keep total response under 150 words. Be conversational, professional."""

        response = self.llm.invoke([HumanMessage(content=prompt)])
        reply = response.content

        self.memory.save_context({'input': ''}, {'output': reply})
        return reply

    def _finalize(self) -> str:
        """Generate the requirements JSON from the full conversation."""
        global AGENT1_OUTPUT, AGENT1_CHAT_HISTORY

        print('\n⚙️  [Agent 1] Generating requirements JSON...', flush=True)

        # Build full conversation summary for the generator
        summary_parts = []
        for phase in SE_PHASES:
            answer = self.phase_answers.get(phase['id'], 'Not provided')
            summary_parts.append(f'=== {phase["label"]} ===\n{answer}')
        conversation_summary = '\n\n'.join(summary_parts)

        gen_prompt = build_generation_prompt(conversation_summary, self.iso_context)

        print('⚙️  [Agent 1] Calling LLM for structured JSON generation...', flush=True)
        response = self.llm.invoke([HumanMessage(content=gen_prompt)])
        raw = response.content.strip()

        # Strip markdown fences if present
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'^```\s*', '', raw)
        raw = re.sub(r'```$', '', raw).strip()

        try:
            AGENT1_OUTPUT = json.loads(raw)

            # Capture chat history for Agent 2 (separately from the JSON schema)
            try:
                history = self.memory.load_memory_variables({}).get('chat_history', [])
                AGENT1_CHAT_HISTORY = [
                    {
                        'role': ('user' if getattr(m, 'type', '') == 'human' else 'assistant'),
                        'content': getattr(m, 'content', '')
                    }
                    for m in history
                    if getattr(m, 'content', '')
                ]
            except Exception:
                AGENT1_CHAT_HISTORY = []

            # Save to disk (timestamped) — keep existing behavior
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            out_path = Path.cwd() / f'requirements_{ts}.json'
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(AGENT1_OUTPUT, f, indent=2, ensure_ascii=False)

            # Save deterministic snapshot for downstream agents
            snapshot_path = Path.cwd() / 'data' / 'engineering_model_iso15926.json'
            snapshot_path.parent.mkdir(parents=True, exist_ok=True)
            with open(snapshot_path, 'w', encoding='utf-8') as f:
                json.dump(AGENT1_OUTPUT, f, indent=2, ensure_ascii=False)

            print(f'✅ [Agent 1] Requirements JSON saved: {out_path}', flush=True)
            print(f'✅ [Agent 1] Snapshot saved: {snapshot_path}', flush=True)
            self.done = True
            return (
                f'✅ **Agent 1 complete!** I have generated your requirements JSON and saved it to:\n'
                f'`{out_path}`\n'
                f'`{snapshot_path}`\n\n'
                f'The JSON contains:\n'
                f'- **{len(AGENT1_OUTPUT.get("iso15926_model",{}).get("possible_individuals",[]))} stakeholder requirements**\n'
                f'- **{len(AGENT1_OUTPUT.get("iso15926_model",{}).get("classes",[]))} operational/lifecycle scenarios**\n'
                f'- **{len(AGENT1_OUTPUT.get("iso15926_model",{}).get("things",[]))} actors/interfaces**\n\n'
                f'👉 **Now run the Research Agent (Agent 2)** in the next cell to begin the research phase!'
            )
        except json.JSONDecodeError as e:
            print(f'⚠️  [Agent 1] JSON parse error: {e}', flush=True)
            AGENT1_OUTPUT = {'raw': raw, 'parse_error': str(e)}
            AGENT1_CHAT_HISTORY = []
            self.done = True
            return f'⚠️ Generated output could not be parsed as JSON. Raw output saved. Error: {e}'


print('✓ OperationalAgent (Agent 1) defined')

✓ OperationalAgent (Agent 1) defined


---
## CELL 5 — Agent 1 Chat Interface (Run This)

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 1 (SIMULATED FOR THIS RUN)
# You asked to NOT run the interactive SE interview right now.
# Instead we assume Agent 1 already ran and produced the snapshot JSON in /data.
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 70)
print('  AGENT 1 — OPERATIONAL AGENT (snapshot mode)')
print('=' * 70)
print('Agent 1 is running...')

# Robust snapshot path resolution (works even if notebook CWD is /notebooks)
SNAPSHOT_NAME = 'engineering_model_iso15926.json'
candidates = [
    Path.cwd() / 'data' / SNAPSHOT_NAME,
    Path.cwd().parent / 'data' / SNAPSHOT_NAME,
    Path.cwd().parent.parent / 'data' / SNAPSHOT_NAME,
    Path(r'c:/Projects/poc_v1/data') / SNAPSHOT_NAME,
]

snapshot = next((p for p in candidates if p.exists()), None)
assert snapshot, (
    'Missing snapshot. Looked in:\n' + '\n'.join(str(p.resolve()) for p in candidates)
)

with open(snapshot, encoding='utf-8') as f:
    AGENT1_OUTPUT = json.load(f)

print('Agent 1 complete (loaded snapshot).')
print(f'Snapshot path: {snapshot.resolve()}')
print('AGENT1_OUTPUT is ready. Starting Agent 2 next.')

# ---- Previous interactive Agent 1 loop (kept for later) ----------------------
# ISO_PATH = None
# for candidate in [
#     Path('engineering_model_iso15926.json'),
#     Path('/mnt/user-data/uploads/engineering_model_iso15926.json'),
# ]:
#     if candidate.exists():
#         ISO_PATH = str(candidate)
#         print(f'✓ Found ISO reference: {ISO_PATH}')
#         break
#
# agent1 = OperationalAgent(iso_reference_path=ISO_PATH)
# print(agent1.greeting())
# while not agent1.done:
#     user_input = input('You: ').strip()
#     if user_input.lower() in ('quit', 'exit', 'q'):
#         break
#     print(f'Agent 1: {agent1.chat(user_input)}')


  AGENT 1 — OPERATIONAL AGENT (snapshot mode)
Agent 1 is running...
Agent 1 complete (loaded snapshot).
Snapshot path: C:\Projects\poc_v1\data\engineering_model_iso15926.json
AGENT1_OUTPUT is ready. Starting Agent 2 next.


---
## CELL 6 — Research Tools (ArXiv + DuckDuckGo)

In [26]:
import httpx, asyncio
from langchain_core.tools import tool


@tool
async def search_web(query: str) -> str:
    """
    Search the web using DuckDuckGo (no API key needed).
    Use for: standards, regulatory documents, vendor pages, technologies.

    Args:
        query: search query string, e.g. "radiation protection standard IAEA occupational exposure"
    """
    try:
        q = urllib.parse.quote_plus(query)
        url = f'https://html.duckduckgo.com/html/?q={q}'
        async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as c:
            resp = await c.get(url, headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'text/html,application/xhtml+xml',
            })
            html = resp.text

        results = []
        blocks = re.split(r'<div class=["\']result["\']', html)
        for block in blocks[1:8]:
            title_m   = re.search(r'<a[^>]+class=["\']result__a["\'][^>]*>(.*?)</a>', block, re.DOTALL)
            url_m     = re.search(r'<a[^>]+href=["\']([^"\'>]+)["\']', block)
            snippet_m = re.search(r'class=["\']result__snippet["\'][^>]*>(.*?)</(?:a|span)>', block, re.DOTALL)
            if title_m and url_m:
                title   = re.sub(r'<[^>]+>', '', title_m.group(1)).strip()
                href    = url_m.group(1)
                snippet = re.sub(r'<[^>]+>', ' ', snippet_m.group(1)).strip() if snippet_m else ''
                snippet = re.sub(r'\s+', ' ', snippet)[:300]
                if href.startswith('//duckduckgo.com/l/?'):
                    uddg = re.search(r'uddg=([^&]+)', href)
                    if uddg: href = urllib.parse.unquote(uddg.group(1))
                if href.startswith('http') and title:
                    results.append({'title': title, 'url': href, 'snippet': snippet})

        if not results:
            return json.dumps({'error': 'No results', 'results': [], 'query': query})
        return json.dumps({'results': results, 'count': len(results)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'results': []})


@tool
async def search_arxiv(query: str) -> str:
    """
    Search ArXiv for academic papers and technical reports.
    Use for: technology implementations, experimental results, research.

    Args:
        query: search query, e.g. "radiation dosimetry machine learning nuclear"
    """
    try:
        q = urllib.parse.quote_plus(query)
        url = f'http://export.arxiv.org/api/query?search_query=all:{q}&max_results=5&sortBy=relevance'
        async with httpx.AsyncClient(timeout=20.0) as c:
            resp = await c.get(url, headers={'User-Agent': 'ResearchAgent/2.0'})
            xml_text = resp.text

        ns = {'a': 'http://www.w3.org/2005/Atom'}
        root = ET.fromstring(xml_text)
        papers = []
        for entry in root.findall('a:entry', ns):
            title   = (entry.findtext('a:title', '', ns) or '').strip().replace('\n', ' ')
            summary = (entry.findtext('a:summary', '', ns) or '').strip().replace('\n', ' ')[:300]
            pid     = entry.findtext('a:id', '', ns)
            pub     = entry.findtext('a:published', '', ns)
            year    = pub[:4] if pub else ''
            authors = [a.findtext('a:name', '', ns) for a in entry.findall('a:author', ns)][:3]
            if title and pid:
                papers.append({'title': title, 'authors': authors, 'abstract': summary, 'url': pid, 'year': year})

        if not papers:
            return json.dumps({'papers': [], 'count': 0, 'message': 'No ArXiv papers found'})
        return json.dumps({'papers': papers, 'count': len(papers)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'papers': []})


@tool
async def search_semantic_scholar(query: str) -> str:
    """Semantic Scholar search (free, no API key)."""
    try:
        url = 'https://api.semanticscholar.org/graph/v1/paper/search'
        params = {
            'query': query,
            'limit': 8,
            'fields': 'title,year,authors,url,abstract,tldr,externalIds'
        }
        async with httpx.AsyncClient(timeout=20.0, follow_redirects=True) as c:
            resp = await c.get(url, params=params, headers={'User-Agent': 'ResearchAgent/2.0'})
            data = resp.json()

        papers = []
        for p in (data or {}).get('data', []) or []:
            title = (p or {}).get('title')
            url_p = (p or {}).get('url')
            if not title:
                continue
            authors = [a.get('name') for a in (p.get('authors') or [])[:4] if isinstance(a, dict)]
            tldr = (p.get('tldr') or {}).get('text') if isinstance(p.get('tldr'), dict) else None
            papers.append({
                'title': title,
                'authors': authors,
                'year': p.get('year'),
                'abstract': (p.get('abstract') or '')[:400],
                'tldr': tldr,
                'url': url_p,
                'external_ids': p.get('externalIds')
            })

        return json.dumps({'papers': papers, 'count': len(papers)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'papers': []})


@tool
async def search_openalex(query: str) -> str:
    """OpenAlex works search (free, no API key)."""
    try:
        url = 'https://api.openalex.org/works'
        params = {'search': query, 'per-page': 8}
        async with httpx.AsyncClient(timeout=20.0, follow_redirects=True) as c:
            resp = await c.get(url, params=params, headers={'User-Agent': 'ResearchAgent/2.0'})
            data = resp.json()

        works = []
        for w in (data or {}).get('results', []) or []:
            title = (w or {}).get('title')
            if not title:
                continue
            works.append({
                'title': title,
                'year': w.get('publication_year'),
                'url': w.get('id'),
                'doi': (w.get('doi') or ''),
                'host_venue': ((w.get('host_venue') or {}).get('display_name') if isinstance(w.get('host_venue'), dict) else None)
            })

        return json.dumps({'works': works, 'count': len(works)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'works': []})


@tool
async def search_crossref(query: str) -> str:
    """Crossref works search (free, no API key)."""
    try:
        url = 'https://api.crossref.org/works'
        params = {'query': query, 'rows': 8}
        async with httpx.AsyncClient(timeout=20.0, follow_redirects=True) as c:
            resp = await c.get(url, params=params, headers={'User-Agent': 'ResearchAgent/2.0'})
            data = resp.json()

        items = []
        for it in (((data or {}).get('message') or {}).get('items') or [])[:8]:
            title = (it.get('title') or [''])[0]
            if not title:
                continue
            items.append({
                'title': title,
                'year': ((it.get('published-print') or it.get('published-online') or {}).get('date-parts') or [[None]])[0][0],
                'url': it.get('URL'),
                'doi': it.get('DOI'),
                'type': it.get('type')
            })

        return json.dumps({'items': items, 'count': len(items)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'items': []})


@tool
async def search_osti(query: str) -> str:
    """OSTI.gov records search (free, no API key)."""
    try:
        url = 'https://www.osti.gov/api/v1/records'
        params = {'q': query, 'page': 0, 'size': 8}
        async with httpx.AsyncClient(timeout=20.0, follow_redirects=True) as c:
            resp = await c.get(url, params=params, headers={'User-Agent': 'ResearchAgent/2.0'})
            data = resp.json()

        records = []
        for r in (data or {}).get('records', []) or []:
            title = r.get('title')
            if not title:
                continue
            records.append({
                'title': title,
                'year': r.get('publication_date', '')[:4] if r.get('publication_date') else None,
                'url': r.get('product_nsti_url') or r.get('osti_id') or r.get('doi') or r.get('landing_page_url'),
                'doi': r.get('doi')
            })

        return json.dumps({'records': records, 'count': len(records)})
    except Exception as exc:
        return json.dumps({'error': str(exc), 'records': []})


@tool
async def build_research_record(record_json: str) -> str:
    """Assemble and validate the final research record for one requirement."""
    try:
        d    = json.loads(record_json)
        stds = d.get('standards', [])
        tech = d.get('technologies', [])
        if stds:
            best = max(stds, key=lambda s: float(s.get('similarity_score', 0)))
            d.setdefault('best_standard', best.get('name'))
            d.setdefault('best_similarity_score', float(best.get('similarity_score', 0)))
        else:
            d.setdefault('best_similarity_score', 0.0)
        if tech:
            d.setdefault('top_technology', tech[0].get('name'))
        sc = float(d.get('best_similarity_score', 0))
        d['gap_severity'] = ('no_match' if not stds else
                             'covered'  if sc >= 0.80 else
                             'partial'  if sc >= 0.50 else 'gap')
        d['researched_at'] = datetime.now(timezone.utc).isoformat()
        return json.dumps({'status': 'ok', 'record': d})
    except Exception as exc:
        return json.dumps({'status': 'error', 'error': str(exc)})


RESEARCH_TOOLS = [
    search_web,
    search_arxiv,
    search_semantic_scholar,
    search_openalex,
    search_crossref,
    search_osti,
    build_research_record,
]

print('✓ Research tools defined:')
print('  - search_web              — DuckDuckGo HTML (fragile fallback)')
print('  - search_arxiv            — ArXiv API')
print('  - search_semantic_scholar — Semantic Scholar API (no key)')
print('  - search_openalex         — OpenAlex API (no key)')
print('  - search_crossref         — Crossref API (no key)')
print('  - search_osti             — OSTI.gov API (no key)')
print('  - build_research_record   — assemble + validate final record JSON')

✓ Research tools defined:
  - search_web              — DuckDuckGo HTML (fragile fallback)
  - search_arxiv            — ArXiv API
  - search_semantic_scholar — Semantic Scholar API (no key)
  - search_openalex         — OpenAlex API (no key)
  - search_crossref         — Crossref API (no key)
  - search_osti             — OSTI.gov API (no key)
  - build_research_record   — assemble + validate final record JSON


---
## CELL 7 — Agent 2: Research Agent

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 2 — RESEARCH AGENT
# Reads Agent 1's JSON, runs a 4-step research pipeline per requirement:
#   Step 2: Validate understanding with user
#   Step 3: Generate search queries & collect candidates
#   Step 4: Filter, score, and rank candidates
# Uses LangChain ConversationBufferMemory across the research session.
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_STEP2 = """You are a research agent analyzing a system. The user has provided a JSON description of the system,
including functions, domain, constraints, and active domains.

Your task:
1. Summarize the system understanding in a concise, structured way.
2. Ask the user to confirm if your understanding is correct.
3. Highlight any uncertain areas (e.g., ambiguous functions, missing constraints, unclear domains).
4. Use clear, structured language for easy validation.
5. Wait for user feedback before proceeding to research.

Return ONLY valid JSON (no markdown, no preamble) in this exact shape:
{
  "function": "...",
  "domain": "...",
  "active_domains": ["..."],
  "constraints": ["..."],
  "uncertainties": ["..."]
}
"""

SYSTEM_PROMPT_STEP3 = """You are a research agent tasked with finding possible technologies, methods, systems, or approaches
to achieve a specific system function.

Input:
- Confirmed system understanding JSON.

Your task:
1. Generate search queries based on:
   - Function
   - Domain
   - Constraints
   - Active domains
2. Return ONLY valid JSON with this exact shape:
{
  "queries": ["...", "..."]
}

Rules:
- Queries must be specific and target reliable sources.
- Avoid science-fiction or impossible solutions.
"""

SYSTEM_PROMPT_STEP3_EXTRACT = """You are a research agent. You are given:
- Confirmed system understanding JSON
- Raw search results (web + ArXiv)

Your task:
1. Extract candidate solutions (technologies, methods, systems, approaches).
2. Return ONLY a JSON array of candidates with fields:
[
  {
    "name": "Solution name",
    "description": "Brief description of the solution",
    "source": "URL or reference",
    "domain": "...",
    "function_alignment": "...",
    "notes": "Additional comments"
  }
]

Rules:
- Do not filter yet; collect plausible candidates.
- Prefer sources from the provided raw results.
- If you must add a candidate from general knowledge, still include a best-effort canonical source URL.
- Never invent a paper id; if you cite ArXiv, it must appear in the raw results.
"""

SYSTEM_PROMPT_STEP4 = """You are a research agent tasked with evaluating a list of candidate solutions for a system function.

Input:
- Confirmed system understanding JSON.
- Candidate solutions JSON array.

Your task:
1. Evaluate each candidate solution according to:
   - Does it achieve the intended function? (Gap analysis)
   - Technology Readiness Level (TRL) if known
   - Feasibility within domain and constraints
   - Realism: is it implementable or science-fiction?
2. Assign a score 0-100 to each candidate based on the criteria above.
3. Return ONLY a filtered and ranked JSON array, sorted by score, with fields:
[
  {
    "name": "Solution name",
    "description": "Brief description",
    "source": "URL or reference (copy from input if present)",
    "domain": "domain tag if known",
    "score": 0-100,
    "TRL": "Low/Medium/High/Unknown",
    "feasibility": "Low/Medium/High",
    "realism": "Realistic/Experimental/Science-Fiction",
    "gap_analysis": "1-2 sentences explaining alignment/gap",
    "notes": "Short explanation of why this candidate was selected or rejected",
    "provenance": "retrieved|llm",
    "verified": true|false|null
  }
]

Rules:
- Keep `source`, `domain`, `provenance`, `verified` from the input candidate if available.
- Be concise and factual.
- Do not invent unsupported details.
"""


def coerce_json(text: str) -> Any:
    raw = (text or '').strip()
    raw = re.sub(r'^```json\s*|^```\s*|```\s*$', '', raw).strip()
    try:
        return json.loads(raw)
    except Exception:
        m = re.search(r'\{[\s\S]*\}|\[[\s\S]*\]', raw)
        if not m:
            raise
        return json.loads(m.group())


def prompt_step2(req_data: Dict, project_meta: Dict, model_meta: Dict) -> str:
    active_domains = (model_meta or {}).get('active_domains') or []
    payload = {
        'project': {
            'project_name': project_meta.get('project_name', ''),
            'domain': project_meta.get('domain', ''),
            'context_summary': project_meta.get('context_summary', ''),
        },
        'requirement': {
            'name': req_data.get('name', ''),
            'description': req_data.get('description', ''),
            'priority': req_data.get('priority', ''),
            'domain_tag': req_data.get('domain_tag', ''),
            'rationale': req_data.get('rationale', ''),
            'section': req_data.get('_section', ''),
            'class': req_data.get('class', ''),
        },
        'active_domains': active_domains,
    }
    return SYSTEM_PROMPT_STEP2 + "\n\nINPUT JSON:\n" + json.dumps(payload, indent=2, ensure_ascii=False)


def prompt_step3_queries(understanding: Dict) -> str:
    return SYSTEM_PROMPT_STEP3 + "\n\nCONFIRMED UNDERSTANDING JSON:\n" + json.dumps(understanding, indent=2, ensure_ascii=False)


def prompt_step3_extract(understanding: Dict, raw_results: Dict) -> str:
    return (
        SYSTEM_PROMPT_STEP3_EXTRACT
        + "\n\nCONFIRMED UNDERSTANDING JSON:\n"
        + json.dumps(understanding, indent=2, ensure_ascii=False)
        + "\n\nRAW SEARCH RESULTS JSON:\n"
        + json.dumps(raw_results, indent=2, ensure_ascii=False)[:12000]
    )


def prompt_step4_enrich(understanding: Dict, candidates: List[Dict]) -> str:
    """Annotate candidates with score/TRL/etc without sorting."""
    enrich_instr = (
        "You MUST preserve the input order (do NOT sort).\n"
        "Return the same number of items as provided.\n"
    )
    return (
        SYSTEM_PROMPT_STEP4
        + "\n\nIMPORTANT:\n"
        + enrich_instr
        + "\nCONFIRMED UNDERSTANDING JSON:\n"
        + json.dumps(understanding, indent=2, ensure_ascii=False)
        + "\n\nCANDIDATE SOLUTIONS JSON:\n"
        + json.dumps(candidates, indent=2, ensure_ascii=False)[:12000]
    )


def prompt_step4_rank(understanding: Dict, candidates: List[Dict]) -> str:
    """Rank candidates (sorted by score descending)."""
    return (
        SYSTEM_PROMPT_STEP4
        + "\n\nCONFIRMED UNDERSTANDING JSON:\n"
        + json.dumps(understanding, indent=2, ensure_ascii=False)
        + "\n\nCANDIDATE SOLUTIONS JSON:\n"
        + json.dumps(candidates, indent=2, ensure_ascii=False)[:12000]
    )


REACT_TEMPLATE = """You are the Research Agent for a Systems Engineering platform.

PROJECT CONTEXT:
{project_context}

REQUIREMENT TO RESEARCH:
  ID: {req_id}
  Statement: {req_statement}
  Domain: {domain}
  Priority: {priority}
  Rationale: {rationale}

MANDATORY STEPS:
1. search_web: search for governing standards and regulations
2. search_arxiv: search for academic papers and technologies
3. search_web: search for vendor implementations and existing solutions
4. build_research_record: assemble the complete research record JSON

For build_research_record, provide:
{{
  "req_id": "{req_id}",
  "req_statement": "{req_statement}",
  "domain_tag": "...",
  "criticality": "high|medium|low",
  "standards": [{{"name":"...","clause":"...","similarity_score":0.0-1.0,
                  "issuing_body":"...","authority_level":"national|international",
                  "source_url":"...","year":"..."}}],
  "technologies": [{{"name":"...","description":"...","source":"...",
                     "trl":"TRL 1-9","feasibility":"Low|Medium|High",
                     "realism":"Realistic|Experimental|Science-Fiction",
                     "score":0-100,"notes":"..."}}],
  "gap_description": "...",
  "recommendation": "..."
}}

Tools available: {tools}
Tool names: {tool_names}

Format EXACTLY:
Thought: <reasoning>
Action: <tool name>
Action Input: <input>
Observation: <result>
... (repeat)
Thought: All steps complete.
Final Answer: <the JSON string from build_research_record>

{agent_scratchpad}"""


def build_step2_summary_prompt(req_data: Dict, project_meta: Dict) -> str:
    
    return f"""You are a research agent. A user has provided a structured systems engineering 
requirements JSON. Your task is to summarize the SPECIFIC requirement below and validate 
your understanding with the user.

PROJECT: {project_meta.get('project_name', 'Unknown')} | Domain: {project_meta.get('domain', 'Unknown')}
CONTEXT: {project_meta.get('context_summary', '')}

REQUIREMENT:
  Name: {req_data.get('name', '')}
  Description: {req_data.get('description', '')}
  Priority: {req_data.get('priority', '')}
  Domain: {req_data.get('domain_tag', '')}
  Rationale: {req_data.get('rationale', '')}

Your task:
1. Summarize your understanding of this requirement concisely
2. Identify any ambiguities or missing information  
3. Ask the user to confirm or correct your understanding before proceeding
4. Be specific — don't be generic. Reference the actual domain and context.

Format your response as a clear summary with bullet points for uncertainties.
End with a direct question asking the user to confirm or provide corrections."""


def build_step4_filter_prompt(understanding_json: str, candidates_json: str) -> str:
    
    return f"""You are a research agent evaluating candidate solutions.

CONFIRMED UNDERSTANDING:
{understanding_json}

CANDIDATE SOLUTIONS FROM SEARCH:
{candidates_json}

Your task:
1. Evaluate each candidate:
   - Does it achieve the intended function? (gap analysis)
   - Technology Readiness Level (TRL 1-9)
   - Feasibility within domain constraints
   - Realism: Realistic / Experimental / Science-Fiction
2. Assign a score 0-100 to each candidate
3. Return a filtered, ranked JSON array sorted by score:

Return ONLY a JSON array (no preamble):
[
  {{
    "name": "...",
    "description": "...",
    "score": 0-100,
    "trl": "TRL N",
    "feasibility": "Low|Medium|High",
    "realism": "Realistic|Experimental|Science-Fiction",
    "notes": "..."
  }}
]"""


class ResearchAgent:
    """
    Agent 2: Research Agent with multi-step pipeline.
    Uses LangChain memory for conversation + ReAct agent for tool use.
    """

    def __init__(self, requirements_json: Dict, seed_chat_history: Optional[List[Dict[str, str]]] = None):
        self.llm = make_llm(temperature=0.2)
        self.memory = ConversationBufferMemory(
            return_messages=True,
            memory_key='chat_history',
            human_prefix='User',
            ai_prefix='ResearchAgent'
        )
        
        # Seed Agent 2 memory with Agent 1 conversation (if provided)
        if seed_chat_history:
            for m in seed_chat_history:
                role = (m or {}).get('role')
                content = (m or {}).get('content', '')
                if not content:
                    continue
                if role == 'user':
                    self.memory.chat_memory.add_message(HumanMessage(content=content))
                else:
                    self.memory.chat_memory.add_message(AIMessage(content=content))

        self.req_json = requirements_json
        self.meta = requirements_json.get('meta', {})
        self.model_data = requirements_json.get('iso15926_model', {})
        self.model_meta = (self.model_data or {}).get('meta', {})
        self.all_records: List[Dict] = []

        # System-level pipeline state (Step 2 → Step 3 → Step 4)
        self.system_step = 'summarize'   # summarize → confirm → research_done
        self.system_understanding: Optional[Dict] = None
        self.system_candidates: Optional[List[Dict]] = None
        self.system_ranked: Optional[List[Dict]] = None

    def _extract_requirements(self) -> List[Dict]:
        """Extract all researchable requirements from Agent 1's JSON."""
        items = []
        for section in ['possible_individuals', 'classes', 'things']:
            for i, item in enumerate(self.model_data.get(section, [])):
                item['_req_id'] = f'REQ-{len(items)+1:03d}'
                item['_section'] = section
                items.append(item)
        return items

    def greeting(self) -> str:
        project = self.meta.get('project_name', 'your project')
        domain  = self.meta.get('domain', 'not_specified')
        return (
            f'Research Agent activated.\n\n'
            f'Loaded Agent 1 JSON for project: {project} (domain: {domain}).\n\n'
            f'Pipeline:\n'
            f'- Step 2: Summarize and validate system understanding\n'
            f'- Step 3: Search and collect candidate approaches with references\n'
            f'- Step 4: Enrich, then rank candidates\n\n'
            f'Type start to generate the Step 2 summary.'
        )

    def _step2_system_summary(self) -> Dict:
        """Step 2 — summarize the entire system JSON and ask for confirmation."""
        # Keep payload bounded so we don't blow context, but still include key fields.
        payload = {
            'meta': (self.req_json or {}).get('meta', {}),
            'iso15926_model_meta': (self.model_meta or {}),
        }
        # Include either instances or ontology summary depending on what's present
        model = (self.req_json or {}).get('iso15926_model', {})
        if isinstance(model, dict):
            if 'possible_individuals' in model or 'classes' in model or 'things' in model:
                payload['model_instance_counts'] = {
                    'possible_individuals': len((model.get('possible_individuals') or [])),
                    'classes': len((model.get('classes') or [])),
                    'things': len((model.get('things') or [])),
                }
                payload['example_items'] = {
                    'possible_individuals': (model.get('possible_individuals') or [])[:3],
                    'classes': (model.get('classes') or [])[:3],
                    'things': (model.get('things') or [])[:3],
                }
            if 'ontology' in model:
                ents = (((model.get('ontology') or {}).get('entity_types')) or {})
                if isinstance(ents, dict):
                    payload['ontology_entity_type_count'] = len(ents)
                    payload['ontology_entity_type_examples'] = list(ents.keys())[:12]

        prompt = SYSTEM_PROMPT_STEP2 + "\n\nINPUT JSON:\n" + json.dumps(payload, indent=2, ensure_ascii=False)
        response = self.llm.invoke([HumanMessage(content=prompt)])
        understanding = coerce_json(response.content)
        if not isinstance(understanding, dict):
            understanding = {
                'function': 'Summarize system from provided JSON',
                'domain': (self.meta or {}).get('domain', 'not_specified'),
                'active_domains': (self.model_meta or {}).get('active_domains', []) or [],
                'constraints': [],
                'uncertainties': ['LLM returned non-dict JSON; please confirm manually.'],
            }

        # Normalize fields
        understanding.setdefault('active_domains', [])
        understanding.setdefault('constraints', [])
        understanding.setdefault('uncertainties', [])

        self.system_understanding = understanding
        self.system_step = 'confirm'
        self.memory.save_context({'input': 'Step2(system): summarize'},
                                  {'output': json.dumps(understanding, indent=2, ensure_ascii=False)})
        return understanding

    def _step3_search(self, req: Dict) -> Dict:
        """Step 3 — ReAct agent searches for standards and technologies."""
        req_id   = req.get('_req_id', 'REQ-???')
        name     = req.get('name', '')
        desc     = req.get('description', '')
        domain   = req.get('domain_tag', self.meta.get('domain', ''))
        priority = req.get('priority', 'medium')
        rationale= req.get('rationale', desc)
        project_ctx = f"{self.meta.get('project_name','')} | {self.meta.get('context_summary','')}"

        prompt = PromptTemplate.from_template(REACT_TEMPLATE)
        react_agent = create_react_agent(llm=self.llm, tools=RESEARCH_TOOLS, prompt=prompt)
        executor = AgentExecutor(
            agent=react_agent, tools=RESEARCH_TOOLS, verbose=True,
            max_iterations=12, handle_parsing_errors=True,
            return_intermediate_steps=True,
        )

        result = executor.invoke({
            'project_context': project_ctx,
            'req_id': req_id,
            'req_statement': f'{name}: {desc[:200]}',
            'domain': domain,
            'priority': priority,
            'rationale': rationale[:200],
        })
        return result

    def _step4_filter(self, raw_record: Dict, req: Dict) -> Dict:
        """Step 4 — Filter and score the found technologies."""
        understanding = json.dumps({
            'function': req.get('name', ''),
            'domain': req.get('domain_tag', ''),
            'description': req.get('description', '')[:300],
            'constraints': [req.get('rationale', '')[:200]],
        }, indent=2)

        candidates = json.dumps(raw_record.get('technologies', []), indent=2)

        if not raw_record.get('technologies'):
            return raw_record  # nothing to filter

        prompt = build_step4_filter_prompt(understanding, candidates)
        response = self.llm.invoke([HumanMessage(content=prompt)])
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw).strip()

        try:
            ranked = json.loads(raw)
            raw_record['technologies'] = ranked
        except Exception:
            pass  # keep original if parse fails

        return raw_record

    def _run_coro(self, coro):
        """Run async code safely in Jupyter or normal Python."""
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(coro)

        # Jupyter: loop is already running → patch to allow nested run_until_complete
        try:
            import nest_asyncio  # type: ignore
            nest_asyncio.apply()
        except Exception as e:
            raise RuntimeError(
                "Jupyter detected an active event loop; install nest_asyncio to run async searches. "
                "Try: pip install nest_asyncio"
            ) from e

        return loop.run_until_complete(coro)

    async def _run_searches(self, queries: List[str]) -> Dict[str, Any]:
        """Run multiple sources for each query (APIs preferred)."""
        out: Dict[str, Any] = {
            'queries': queries,
            'web': [],
            'arxiv': [],
            'semantic_scholar': [],
            'openalex': [],
            'crossref': [],
            'osti': [],
        }
        for idx, q in enumerate(queries[:6], 1):
            print(f'Step 3b: Searching ({idx}/{min(len(queries),6)}): {q}', flush=True)

            try:
                r = json.loads(await search_arxiv.ainvoke(q))
                out['arxiv'].append({'query': q, 'results': r})
                print(f"  - ArXiv: ok ({len((r or {}).get('papers', []) or [])})", flush=True)
            except Exception as e:
                out['arxiv'].append({'query': q, 'error': str(e)})
                print(f"  - ArXiv: error ({e})", flush=True)

            try:
                r = json.loads(await search_semantic_scholar.ainvoke(q))
                out['semantic_scholar'].append({'query': q, 'results': r})
                print(f"  - Semantic Scholar: ok ({len((r or {}).get('papers', []) or [])})", flush=True)
            except Exception as e:
                out['semantic_scholar'].append({'query': q, 'error': str(e)})
                print(f"  - Semantic Scholar: error ({e})", flush=True)

            try:
                r = json.loads(await search_openalex.ainvoke(q))
                out['openalex'].append({'query': q, 'results': r})
                print(f"  - OpenAlex: ok ({len((r or {}).get('works', []) or [])})", flush=True)
            except Exception as e:
                out['openalex'].append({'query': q, 'error': str(e)})
                print(f"  - OpenAlex: error ({e})", flush=True)

            try:
                r = json.loads(await search_crossref.ainvoke(q))
                out['crossref'].append({'query': q, 'results': r})
                print(f"  - Crossref: ok ({len((r or {}).get('items', []) or [])})", flush=True)
            except Exception as e:
                out['crossref'].append({'query': q, 'error': str(e)})
                print(f"  - Crossref: error ({e})", flush=True)

            try:
                r = json.loads(await search_osti.ainvoke(q))
                out['osti'].append({'query': q, 'results': r})
                print(f"  - OSTI: ok ({len((r or {}).get('records', []) or [])})", flush=True)
            except Exception as e:
                out['osti'].append({'query': q, 'error': str(e)})
                print(f"  - OSTI: error ({e})", flush=True)

            # Optional fallback for vendors/regs
            try:
                r = json.loads(await search_web.ainvoke(q))
                out['web'].append({'query': q, 'results': r})
                print(f"  - Web (DDG): ok ({len((r or {}).get('results', []) or [])})", flush=True)
            except Exception as e:
                out['web'].append({'query': q, 'error': str(e)})
                print(f"  - Web (DDG): error ({e})", flush=True)

        return out

    def _collect_allowed_sources(self, raw_results: Dict[str, Any]) -> set:
        allowed = set()

        def _add_urls(items, key='url'):
            for it in items or []:
                u = (it or {}).get(key)
                if isinstance(u, str) and u.startswith('http'):
                    allowed.add(u)

        # DuckDuckGo
        for block in (raw_results or {}).get('web', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('results', []), key='url')

        # ArXiv
        for block in (raw_results or {}).get('arxiv', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('papers', []), key='url')

        # Semantic Scholar
        for block in (raw_results or {}).get('semantic_scholar', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('papers', []), key='url')

        # OpenAlex
        for block in (raw_results or {}).get('openalex', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('works', []), key='url')

        # Crossref
        for block in (raw_results or {}).get('crossref', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('items', []), key='url')

        # OSTI
        for block in (raw_results or {}).get('osti', []):
            results = (block or {}).get('results', {})
            _add_urls((results or {}).get('records', []), key='url')

        return allowed

    async def _verify_url(self, url: str, name_hint: str = '') -> Dict[str, Any]:
        """Fetch URL and return verification evidence."""
        try:
            async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as c:
                resp = await c.get(url, headers={'User-Agent': 'ResearchAgent/2.0'})
            text = resp.text or ''
            ok = (200 <= int(resp.status_code) < 400)
            # basic evidence: does page mention the candidate name (very rough)
            hit = False
            if name_hint:
                hit = name_hint.lower()[:40] in text.lower()
            return {
                'url': url,
                'status_code': int(resp.status_code),
                'verified': bool(ok and (hit or not name_hint)),
                'name_match': bool(hit),
                'evidence': (text[:300] if ok else '')
            }
        except Exception as e:
            return {'url': url, 'verified': False, 'error': str(e)}

    def _annotate_provenance(self, candidates: List[Dict], allowed_sources: set) -> List[Dict]:
        out = []
        for c in candidates:
            if not isinstance(c, dict):
                continue
            src = (c.get('source') or '').strip()
            prov = 'retrieved' if src in allowed_sources else 'llm'
            c2 = dict(c)
            c2['provenance'] = prov
            c2['verified'] = None
            out.append(c2)
        return out

    def _verify_candidates(self, candidates: List[Dict]) -> List[Dict]:
        # verify only top N to keep runtime reasonable
        to_check = [c for c in candidates if isinstance(c, dict) and isinstance(c.get('source'), str) and c.get('source','').startswith('http')]
        to_check = to_check[:15]
        async def _run():
            out = []
            for c in to_check:
                v = await self._verify_url(c.get('source',''), c.get('name',''))
                out.append((c.get('source',''), v))
            return out
        results = self._run_coro(_run())
        by_url = {u: v for (u, v) in results}

        final = []
        for c in candidates:
            if not isinstance(c, dict):
                continue
            src = (c.get('source') or '').strip()
            v = by_url.get(src)
            c2 = dict(c)
            if v:
                c2['verified'] = bool(v.get('verified'))
                c2['verification'] = v
            else:
                # no verification attempted
                c2['verified'] = False if c2.get('provenance') == 'retrieved' else None
            final.append(c2)
        return final

    def _step3_system_collect_candidates(self) -> List[Dict]:
        """Step 3 — generate queries, run searches, extract candidate solutions."""
        assert self.system_understanding, 'Step 2 must be confirmed first.'

        q_prompt = prompt_step3_queries(self.system_understanding)
        q_resp = self.llm.invoke([HumanMessage(content=q_prompt)])
        q_json = coerce_json(q_resp.content)
        queries = (q_json or {}).get('queries', []) if isinstance(q_json, dict) else []
        if not queries:
            fn = (self.system_understanding or {}).get('function', '')
            dom = (self.system_understanding or {}).get('domain', '')
            queries = [f"{fn} {dom} standards", f"{fn} {dom} technology", f"{fn} {dom} vendor solution"]

        queries = [str(q) for q in queries if q]
        print('Step 3a: Generated search queries', flush=True)
        for i, q in enumerate(queries, 1):
            print(f'  {i:02d}. {q}', flush=True)

        raw_results = self._run_coro(self._run_searches(queries))
        allowed_sources = self._collect_allowed_sources(raw_results)

        # quick counts summary
        def _count(path: str, key: str) -> int:
            n = 0
            for block in (raw_results or {}).get(path, []):
                results = (block or {}).get('results', {})
                arr = (results or {}).get(key, []) or []
                n += len(arr)
            return n

        print('Step 3b: Retrieved items (totals)', flush=True)
        print(f"  - ArXiv papers          : {_count('arxiv','papers')}", flush=True)
        print(f"  - Semantic Scholar papers: {_count('semantic_scholar','papers')}", flush=True)
        print(f"  - OpenAlex works        : {_count('openalex','works')}", flush=True)
        print(f"  - Crossref items        : {_count('crossref','items')}", flush=True)
        print(f"  - OSTI records          : {_count('osti','records')}", flush=True)
        print(f"  - Web results (DDG)     : {_count('web','results')}", flush=True)

        # 3c-1) Extract candidates grounded in retrieved results
        extract_prompt = prompt_step3_extract(self.system_understanding, raw_results)
        extract_resp = self.llm.invoke([HumanMessage(content=extract_prompt)])
        retrieved_candidates = coerce_json(extract_resp.content)
        if not isinstance(retrieved_candidates, list):
            retrieved_candidates = []

        # 3c-2) Also ask LLM for additional candidates from its own knowledge
        knowledge_prompt = (
            "You are a research agent.\n"
            "Generate additional candidate solutions from your own knowledge that are likely relevant.\n\n"
            "INPUT (confirmed understanding JSON):\n"
            + json.dumps(self.system_understanding, indent=2, ensure_ascii=False)
            + "\n\nReturn ONLY a JSON array with the same candidate schema as Step 3:\n"
            "[{'name':..., 'description':..., 'source':..., 'domain':..., 'function_alignment':..., 'notes':...}]\n\n"
            "Rules:\n"
            "- Provide a best-effort canonical source URL for each candidate (publisher/regulator/vendor/official page).\n"
            "- Do NOT invent ArXiv IDs.\n"
            "- It is OK if the source is not in retrieved results; these will be labeled as LLM provenance.\n"
        )
        knowledge_resp = self.llm.invoke([HumanMessage(content=knowledge_prompt)])
        llm_candidates = coerce_json(knowledge_resp.content)
        if not isinstance(llm_candidates, list):
            llm_candidates = []

        # Merge + de-dupe by source then name
        merged: List[Dict] = []
        seen = set()
        for c in (retrieved_candidates + llm_candidates):
            if not isinstance(c, dict):
                continue
            src = (c.get('source') or '').strip()
            name = (c.get('name') or '').strip()
            key = (src.lower() if src else name.lower())
            if not key or key in seen:
                continue
            seen.add(key)
            merged.append(c)

        candidates = self._annotate_provenance(merged, allowed_sources)
        candidates = self._verify_candidates(candidates)

        n_retrieved = sum(1 for c in candidates if isinstance(c, dict) and c.get('provenance') == 'retrieved')
        n_llm = sum(1 for c in candidates if isinstance(c, dict) and c.get('provenance') == 'llm')
        n_verified = sum(1 for c in candidates if isinstance(c, dict) and c.get('verified') is True)
        print('Step 3c: Candidates built', flush=True)
        print(f'  - from retrieved sources: {n_retrieved}', flush=True)
        print(f'  - from LLM knowledge    : {n_llm}', flush=True)
        print(f'  - verified URLs         : {n_verified}', flush=True)

        self.system_candidates = candidates
        self.memory.save_context({'input': 'Step3(system): collect candidates'},
                                  {'output': json.dumps({'count': len(candidates)}, indent=2)})
        return candidates

    def _step4_system_enrich(self) -> List[Dict]:
        """Step 4a — enrich candidates with score/TRL/etc (no sorting)."""
        assert self.system_understanding, 'Missing system understanding.'
        assert self.system_candidates is not None, 'Run Step 3 first.'

        prompt = prompt_step4_enrich(self.system_understanding, self.system_candidates)
        resp = self.llm.invoke([HumanMessage(content=prompt)])
        enriched = coerce_json(resp.content)
        if not isinstance(enriched, list):
            enriched = []

        # Ensure we keep provenance/verified/source from Step 3 if missing
        by_src = {}
        for c in (self.system_candidates or []):
            if isinstance(c, dict):
                s = (c.get('source') or '').strip()
                if s:
                    by_src[s] = c

        fixed = []
        for r in enriched:
            if not isinstance(r, dict):
                continue
            src = (r.get('source') or '').strip()
            base = by_src.get(src, {}) if src else {}
            rr = dict(base)
            rr.update(r)
            rr.setdefault('provenance', base.get('provenance'))
            rr.setdefault('verified', base.get('verified'))
            fixed.append(rr)

        self.system_enriched = fixed
        return fixed

    def _step4_system_rank(self) -> List[Dict]:
        """Step 4b — rank candidates using the Step 4 system prompt."""
        assert self.system_understanding, 'Missing system understanding.'
        assert self.system_candidates is not None, 'Run Step 3 first.'

        prompt = prompt_step4_rank(self.system_understanding, self.system_candidates)
        resp = self.llm.invoke([HumanMessage(content=prompt)])
        ranked = coerce_json(resp.content)
        if not isinstance(ranked, list):
            ranked = []

        # Post-process: compute rollup summary
        def _bucket(score: Any) -> str:
            try:
                s = float(score)
            except Exception:
                return 'unknown'
            if s >= 80:
                return 'deploy_now'
            if s >= 50:
                return 'monitor'
            return 'rd_invest'

        domain_counts: Dict[str, Dict[str, Any]] = {}
        buckets = {'deploy_now': 0, 'monitor': 0, 'rd_invest': 0, 'unknown': 0}
        for r in ranked:
            if not isinstance(r, dict):
                continue
            b = _bucket(r.get('score'))
            buckets[b] = buckets.get(b, 0) + 1
            dom = (r.get('domain') or 'unknown')
            if not isinstance(dom, str):
                dom = 'unknown'
            domain_counts.setdefault(dom, {'count': 0, 'min': None, 'max': None})
            domain_counts[dom]['count'] += 1
            try:
                s = float(r.get('score'))
                domain_counts[dom]['min'] = s if domain_counts[dom]['min'] is None else min(domain_counts[dom]['min'], s)
                domain_counts[dom]['max'] = s if domain_counts[dom]['max'] is None else max(domain_counts[dom]['max'], s)
            except Exception:
                pass

        summary = {'domain_coverage': domain_counts, 'buckets': buckets}

        self.system_ranked = ranked
        self.system_step = 'research_done'
        self.all_records.append({
            'type': 'system_research',
            'understanding': self.system_understanding,
            'candidates': self.system_candidates,
            'enriched_pre_rank': getattr(self, 'system_enriched', None),
            'ranked': ranked,
            'summary': summary,
        })
        self.system_summary = summary
        return ranked

    def chat(self, user_input: str) -> str:
        """System-level Step 2 → Step 3 → Step 4 chat flow."""
        ui = (user_input or '').strip()
        low = ui.lower()

        if low in ('quit', 'exit', 'q'):
            return 'Stopping Agent 2.'

        # Step 2: summarize system JSON
        if low in ('start', 'summarize') and self.system_step == 'summarize':
            understanding = self._step2_system_summary()
            pretty = json.dumps(understanding, indent=2, ensure_ascii=False)
            return (
                'Step 2: System understanding (please confirm)\n\n'
                f'{pretty}\n\n'
                'Type yes to confirm, or type corrections.'
            )

        # User corrections during confirm
        if self.system_step == 'confirm' and low not in ('yes', 'y', 'confirmed', 'ok', 'proceed'):
            if self.system_understanding is None:
                return 'Type `start` to generate the Step 2 system summary.'
            self.system_understanding['user_corrections'] = ui
            self.memory.save_context({'input': f'Correction(system): {ui}'}, {'output': 'Noted.'})
            pretty = json.dumps(self.system_understanding, indent=2, ensure_ascii=False)
            return (
                'Noted. Updated understanding (corrections recorded).\n\n'
                f'{pretty}\n\n'
                'Type yes to confirm and start Step 3 research.'
            )

        # Confirm → Step 3 + Step 4a (enrich only; no ranking)
        if self.system_step == 'confirm' and low in ('yes', 'y', 'confirmed', 'ok', 'proceed'):
            print('\nStep 3: Generating queries and searching...', flush=True)
            candidates = self._step3_system_collect_candidates()

            print('Step 4: Enriching candidates (no ranking)...', flush=True)
            enriched = self._step4_system_enrich()

            # Save full enriched output to disk (so you always get all results)
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            out_path = Path.cwd() / f'candidates_enriched_{ts}.json'
            with open(out_path, 'w', encoding='utf-8') as f:
                json.dump(enriched, f, indent=2, ensure_ascii=False)

            self.system_step = 'research_done'
            self.all_records.append({
                'type': 'system_candidates_enriched',
                'understanding': self.system_understanding,
                'candidates': candidates,
                'enriched': enriched,
                'saved_to': str(out_path),
            })

            # Print all results in a readable way
            return (
                'Enriched candidates (no ranking)\n'
                f'- Candidates collected: {len(candidates)}\n'
                f'- Enriched returned: {len(enriched)}\n'
                f'- Saved to: {out_path}\n\n'
                + json.dumps(enriched, indent=2, ensure_ascii=False)
                + '\n\nType save to write session results JSON.'
            )

        if low == 'save':
            return self._finalize()

        if self.system_step == 'research_done':
            return 'Research already complete. Type `save` to write results JSON.'

        return 'Type `start` to generate the Step 2 system summary.'

    def _parse_agent_output(self, output: str) -> Dict:
        """Extract the research record JSON from agent output."""
        # Try direct JSON parse
        try:
            d = json.loads(output)
            return d.get('record', d)
        except Exception:
            pass
        # Try extracting JSON from text
        match = re.search(r'\{[\s\S]+\}', output)
        if match:
            try:
                d = json.loads(match.group())
                return d.get('record', d)
            except Exception:
                pass
        return {'raw_output': output[:500], 'parse_error': True}

    def _research_all(self) -> str:
        """Auto-research all requirements without user confirmation per item."""
        total = len(self.all_requirements)
        print(f'\n🚀 [Agent 2] Auto-researching all {total} requirements...', flush=True)

        for i, req in enumerate(self.all_requirements):
            print(f'\n[{i+1}/{total}] Researching: {req.get("name","")}', flush=True)
            try:
                result  = self._step3_search(req)
                parsed  = self._parse_agent_output(result.get('output', '{}'))
                parsed  = self._step4_filter(parsed, req)
                self.all_records.append(parsed)
                print(f'  ✓ Done — gap: {parsed.get("gap_severity","?")} | techs: {len(parsed.get("technologies",[]))}', flush=True)
            except Exception as e:
                print(f'  ✗ Error: {e}', flush=True)
                self.all_records.append({'req_id': req.get('_req_id'), 'error': str(e)})

        self.current_req_idx = total
        return self._finalize()

    def _finalize(self) -> str:
        """Save all research results to disk."""
        global AGENT2_RECORDS
        AGENT2_RECORDS = self.all_records

        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        out = {
            'generated_at': datetime.now(timezone.utc).isoformat(),
            'project': self.meta,
            'total_researched': len(self.all_records),
            'records': self.all_records
        }
        out_path = Path.cwd() / f'research_results_{ts}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(out, f, indent=2, ensure_ascii=False)

        covered = sum(1 for r in self.all_records if r.get('gap_severity') == 'covered')
        partial = sum(1 for r in self.all_records if r.get('gap_severity') == 'partial')
        gaps    = sum(1 for r in self.all_records if r.get('gap_severity') == 'gap')

        return (
            f'\nResearch complete\n'
            f'- Total researched: {len(self.all_records)}\n'
            f'- Covered: {covered} | Partial: {partial} | Gaps: {gaps}\n'
            f'- Results saved to: {out_path}'
        )


print('✓ ResearchAgent (Agent 2) defined')

✓ ResearchAgent (Agent 2) defined


In [28]:
!!pip install nest_asyncio

['Requirement already satisfied: nest_asyncio in c:\\projects\\poc\\venv\\lib\\site-packages (1.6.0)',
 '',
 '[notice] A new release of pip is available: 24.0 -> 26.0.1',
 '[notice] To update, run: C:\\Projects\\poc\\venv\\Scripts\\python.exe -m pip install --upgrade pip']

---
## CELL 8 — Agent 2 Chat Interface (Run After Agent 1 is Done)

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT 2 CHAT LOOP
# Requires AGENT1_OUTPUT to be set (run Agent 1 first).
# Or load from a saved JSON file.
# ─────────────────────────────────────────────────────────────────────────────

# Option A: use Agent 1's live output
source = AGENT1_OUTPUT

# Option B: prefer deterministic snapshot from Agent 1
if not source:
    SNAPSHOT_NAME = 'engineering_model_iso15926.json'
    candidates = [
        Path.cwd() / 'data' / SNAPSHOT_NAME,
        Path.cwd().parent / 'data' / SNAPSHOT_NAME,
        Path.cwd().parent.parent / 'data' / SNAPSHOT_NAME,
        Path(r'c:/Projects/poc_v1/data') / SNAPSHOT_NAME,
    ]
    snapshot = next((p for p in candidates if p.exists()), None)
    if snapshot:
        with open(snapshot, encoding='utf-8') as f:
            source = json.load(f)
        print(f'✓ Loaded Agent 1 snapshot: {snapshot.resolve()}')
    else:
        print('⚠ No Agent 1 snapshot found. Looked in:')
        for p in candidates:
            try:
                print('  -', p.resolve())
            except Exception:
                print('  -', p)

# Option C: load from file (uncomment to use)
# with open('requirements_YYYYMMDD_HHMMSS.json') as f:
#     source = json.load(f)

# Option D: use the uploaded reference JSON for demo
if not source:
    for candidate in [
        Path('engineering_model_iso15926.json'),
        Path('/mnt/user-data/uploads/engineering_model_iso15926.json'),
    ]:
        if candidate.exists():
            with open(candidate, encoding='utf-8') as f:
                raw_demo = json.load(f)
            # Wrap it in the expected schema
            source = {
                'meta': {
                    'project_name': 'Nuclear Reprocessing Demo',
                    'domain': 'nuclear_fuel_reprocessing',
                    'context_summary': 'PUREX nuclear fuel reprocessing plant, France.',
                },
                'iso15926_model': raw_demo.get('iso15926_model', raw_demo)
            }
            print(f'✓ Loaded demo data from {candidate}')
            break

assert source, 'No source data. Run Agent 1 first or set source manually.'

# ── Start Agent 2 ─────────────────────────────────────────────────────────
agent2 = ResearchAgent(source, seed_chat_history=AGENT1_CHAT_HISTORY or None)

print('=' * 70)
print('  AGENT 2 — RESEARCH AGENT  (type "quit" to stop early)')
print('=' * 70)
print()
print(agent2.greeting())
print()

# Auto-run Step 2 summarization for the first item
print('---')
print('Auto-start: Step 2 summarization (from Agent 1 )')
print('---')
first = agent2.chat('start')
print(f'Agent 2: {first}')
print()

# ── Interactive chat loop ──────────────────────────────────────────────────
while True:
    try:
        user_input = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[Interrupted]')
        break

    if not user_input:
        continue
    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Stopping Agent 2 early.')
        break

    print()
    reply = agent2.chat(user_input)
    print(f'Agent 2: {reply}')
    print()

    if user_input.lower() == 'save':
        break

print()
print(f'Agent 2 session ended. Records collected: {len(agent2.all_records)}')

  AGENT 2 — RESEARCH AGENT  (type "quit" to stop early)

Research Agent activated.

Loaded Agent 1 JSON for project: your project (domain: not_specified).

Pipeline:
- Step 2: Summarize and validate system understanding
- Step 3: Search and collect candidate approaches with references
- Step 4: Enrich, then rank candidates

Type start to generate the Step 2 summary.

---
Auto-start: Step 2 summarization (from Agent 1 )
---
Agent 2: Step 2: System understanding (please confirm)

{
  "function": "Nuclear fuel reprocessing system for extracting fissile material (uranium and plutonium) from spent nuclear fuel waste, capable of processing multiple fuel types (UOX, MOX, ERU/REU) with 1800 tHM/year capacity",
  "domain": "Nuclear fuel cycle - spent fuel reprocessing and fissile material recovery",
  "active_domains": [
    "Nuclear engineering",
    "Radiological safety",
    "Process engineering",
    "Regulatory compliance",
    "Quality control"
  ],
  "constraints": [
    "ALARA radiation

---
## CELL 9 — View Results Summary

In [30]:
records = agent2.all_records if 'agent2' in dir() else AGENT2_RECORDS

if not records:
    print('No records yet. Run Agent 2 first.')
else:
    # Find the latest system-level enriched candidates run
    enriched_runs = [r for r in records if isinstance(r, dict) and r.get('type') == 'system_candidates_enriched']
    if not enriched_runs:
        print('No enriched candidate results found in this session.')
        print('Run Agent 2, confirm Step 2, then type yes to generate enriched candidates.')
    else:
        latest = enriched_runs[-1]
        enriched = latest.get('enriched') or []
        saved_to = latest.get('saved_to')

        print('RESEARCH RESULTS SUMMARY')
        print(f"- Enriched candidates: {len(enriched)}")
        if saved_to:
            print(f"- Saved to: {saved_to}")
        print()

        rows = []
        for c in enriched:
            if not isinstance(c, dict):
                continue
            rows.append({
                'name': c.get('name'),
                'domain': c.get('domain'),
                'score': c.get('score'),
                'TRL': c.get('TRL'),
                'feasibility': c.get('feasibility'),
                'realism': c.get('realism'),
                'provenance': c.get('provenance'),
                'verified': c.get('verified'),
                'source': c.get('source'),
                'gap_analysis': (c.get('gap_analysis') or '')[:140],
            })

        # Prefer a clean table if pandas is available
        try:
            import pandas as pd  # type: ignore

            df = pd.DataFrame(rows)
            if not df.empty:
                pd.set_option('display.max_rows', 200)
                pd.set_option('display.max_colwidth', 120)
                display(df)
            else:
                print('No rows to display.')
        except Exception:
            # Fallback: plain text table
            cols = ['name','domain','score','TRL','feasibility','realism','provenance','verified','source']
            print(' | '.join(cols))
            print('-' * 120)
            for r in rows:
                print(' | '.join(str(r.get(k,''))[:40] for k in cols))
            print()

RESEARCH RESULTS SUMMARY
- Enriched candidates: 12
- Saved to: c:\Projects\poc_v1\candidates_enriched_20260318_120926.json

name | domain | score | TRL | feasibility | realism | provenance | verified | source
------------------------------------------------------------------------------------------------------------------------
PUREX Process | Nuclear fuel reprocessing - solvent extr | 95 | High | High | Realistic | retrieved | False | https://openalex.org/W2049334543
Hydrometallurgical Reprocessing | Nuclear fuel reprocessing - aqueous proc | 75 | Medium | Medium | Realistic | retrieved | False | https://www.semanticscholar.org/paper/b8
Spent Fuel Dissolution and Reprocessing  | Nuclear fuel reprocessing - head-end pro | 85 | High | High | Realistic | llm | False | https://openalex.org/W1967989107
COEX Process | Nuclear fuel reprocessing - advanced sol | 80 | Medium | High | Realistic | llm | False | https://www.iaea.org/publications/10884/
GANEX Process | Nuclear fuel reprocessing - 